# 07 - Real-data Evidence Protocol

This notebook applies the frozen `body-weight-evidence-protocol-v1` before any real performance is inspected. It does not tune features, horizons, thresholds, Ridge alpha, or selection rules. REAL evidence and SYNTHETIC pipeline validation are always shown in separate sections.

In [ ]:
import matplotlib.pyplot as plt
from metaform_ds.config import DataScienceConfig
from metaform_ds.modeling.weight import FrozenWeightEvidenceProtocol, render_real_evidence_report
from metaform_ds.real_evidence import RealEvidenceExecutionError, run_real_evidence, run_synthetic_protocol_validation
settings = DataScienceConfig.from_env()
protocol = FrozenWeightEvidenceProtocol()
print(f'Protocol version: {protocol.protocol_version}')
print(f'Protocol hash: {protocol.protocol_hash}')
print(protocol.canonical_json())

## Frozen objective, leakage controls, and thresholds

The unchanged objective predicts an observed body weight near 7 days ahead (plus or minus 2 days) after 5 prior measurements. Expanding-window folds train only on labels observed before each prediction origin. Rolling, lag, cadence, interval, and segment inputs are past-only. Pre-registered evidence requires 30 predictions, 3 eligible users, 2 users with at least 3 predictions, a 60-day span, and at least two cadence and history buckets.

In [ ]:
real_result = None
try:
    real_result = run_real_evidence(settings, protocol)
except RealEvidenceExecutionError as error:
    print(f'REAL MODE ERROR: {error}')
if real_result is not None:
    print(f'REAL MODE STATUS: {real_result.data_origin}')
    print(f'Evidence state: {real_result.evidence_state}')
    print(f'Evidence scope: {real_result.evidence_scope}')
    print(f'Integration review allowed: {real_result.integration_review_allowed}')
    print(real_result.aggregate_summary.to_dict())

## REAL aggregate quality and eligibility

No user IDs, anonymous IDs, or row-level weights are displayed or written. `PERSONAL_TEMPORAL_VALIDATION` permits an honest within-user portfolio analysis but is not multi-user generalization.

In [ ]:
if real_result is None:
    print('REAL evaluation could not run; no synthetic fallback was used.')
else:
    print('Quality:', real_result.quality)
    print('Eligibility:', real_result.eligibility)
    print('Gate reasons:', real_result.reason_codes)

## Frozen REAL evaluation

Metrics appear only when quality and chronological eligibility allow evaluation. Sensitivity analysis remains descriptive and cannot mutate the frozen defaults.

In [ ]:
real_robustness = None if real_result is None else real_result.robustness
if real_robustness is None:
    print('REAL frozen evaluation pending or blocked by quality/eligibility.')
else:
    print(real_robustness.comparison.to_string(index=False))
    print('\nTemporal stability:')
    print(real_robustness.stability.to_string(index=False))
    print('\nCadence coverage:')
    print(real_robustness.cadence_metrics.to_string(index=False))
    print('\nHistory coverage:')
    print(real_robustness.history_metrics.to_string(index=False))
    print('\nIntervals:')
    print(real_robustness.interval_diagnostics.to_string(index=False))

In [ ]:
if real_robustness is not None and not real_robustness.comparison.empty:
    view = real_robustness.comparison.sort_values('mae_kg')
    fig, ax = plt.subplots()
    ax.bar(view['model'], view['mae_kg'])
    ax.set(title='REAL aggregate MAE by frozen method', xlabel='Method', ylabel='MAE (kg)')
    ax.tick_params(axis='x', rotation=20)
    plt.show()
else:
    print('No REAL model-comparison chart: evaluation is pending.')

In [ ]:
if real_robustness is not None and not real_robustness.fold_metrics.empty:
    fig, ax = plt.subplots()
    for model, group in real_robustness.fold_metrics.groupby('model'):
        ax.plot(group['prediction_date'], group['mae_kg'], marker='o', label=model)
    ax.set(title='REAL temporal fold MAE', xlabel='Prediction date (UTC)', ylabel='MAE (kg)')
    ax.legend()
    fig.autofmt_xdate()
    plt.show()
else:
    print('No REAL temporal-stability chart: evaluation is pending.')

In [ ]:
if real_robustness is not None and not real_robustness.predictions.empty:
    cadence_counts = real_robustness.predictions['cadence_bucket'].value_counts().sort_index()
    fig, ax = plt.subplots()
    ax.bar(cadence_counts.index, cadence_counts.values)
    ax.set(title='REAL prediction count by cadence bucket', xlabel='Past-only cadence', ylabel='Predictions')
    plt.show()
else:
    print('No REAL cadence chart: evaluation is pending.')

In [ ]:
if real_robustness is not None and not real_robustness.predictions.empty:
    history_counts = real_robustness.predictions['history_bucket'].value_counts().sort_index()
    fig, ax = plt.subplots()
    ax.bar(history_counts.index, history_counts.values)
    ax.set(title='REAL prediction count by history bucket', xlabel='Prior-history bucket', ylabel='Predictions')
    plt.show()
else:
    print('No REAL history chart: evaluation is pending.')

## Separate SYNTHETIC protocol validation

The following run validates mechanics only. Its accuracy is not REAL evidence, cannot permit integration review, and is never merged with PostgreSQL rows.

In [ ]:
synthetic_result = run_synthetic_protocol_validation(settings, protocol)
print('SYNTHETIC PIPELINE VALIDATION - NOT REAL PERFORMANCE')
print(synthetic_result.aggregate_summary.to_dict())
if synthetic_result.robustness is not None:
    print(synthetic_result.robustness.comparison.to_string(index=False))
assert synthetic_result.data_origin == 'SYNTHETIC'
assert not synthetic_result.integration_review_allowed

## Decision and limitations

An allowed gate means only that evidence is sufficient for a future integration review. It never means production readiness or automatic integration. Missing real data, limited longitudinal history, user diversity, segment coverage, temporal instability, or critical quality issues all block the gate. No model is persisted, served, or consumed by the application.

In [ ]:
if real_result is not None:
    print(render_real_evidence_report(real_result))
    print(f'Enough evidence for next decision gate: {real_result.integration_review_allowed}')
else:
    print('Enough evidence for next decision gate: False')